# Path Expansion Analysis

Analyzes computational paths through the residual stream. Goes beyond component-level analysis to understand multi-hop information flow.

**References:**
- Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"
- Conmy et al. (2023) "Towards Automated Circuit Discovery"

## Why This Matters

Path expansion enumerates and quantifies the computational paths through the network — from specific input tokens through attention heads and MLPs to specific output logits. This makes the implicit computation graph of a transformer explicit and measurable.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Wang et al. (2023) "Interpretability in the Wild: IOI"](https://arxiv.org/abs/2211.00593) — Detailed circuit analysis of indirect object identification

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.path_expansion import (
    enumerate_paths,
    path_contribution_matrix,
    virtual_weight_path,
    residual_stream_decomposition,
    path_patching_matrix,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.array([0, 5, 10, 15, 20, 25, 30, 35])
def metric(logits): return float(logits[-1, 0])
print(f"Model: {cfg.n_layers} layers, {cfg.n_heads} heads")

In [ ]:
# 1. Enumerate computational paths
paths = enumerate_paths(model, tokens, max_depth=2, top_k=5)
print(f"Paths enumerated: {paths['n_paths_enumerated']}")
print(f"\nTop {len(paths['top_paths'])} paths:")
for path, contrib in paths['top_paths']:
    labels = [f"{'L'+str(p[1])+'H'+str(p[2]) if p[0]=='attn' else 'L'+str(p[1])+'MLP'}" for p in path]
    print(f"  {' -> '.join(labels)}: contribution={contrib:.4f}")

In [ ]:
# 2. Path contribution matrix
contrib = path_contribution_matrix(model, tokens)
print(f"Dominant component: {contrib['dominant_component']}")
print(f"Embed contribution: {contrib['embed_contribution']:.4f}")
print(f"\nAttention contributions (per head):")
for l in range(cfg.n_layers):
    vals = [f"{contrib['attn_contributions'][l, h]:.3f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {vals}")
print(f"\nMLP contributions: {[f'{c:.3f}' for c in contrib['mlp_contributions']]}")

In [ ]:
# 3. Virtual weight paths (composition scores)
for (la, ha, lb, hb) in [(0,0,1,0), (0,1,1,1), (0,0,2,0)]:
    vw = virtual_weight_path(model, la, ha, lb, hb)
    print(f"L{la}H{ha} -> L{lb}H{hb}: OV-OV={vw['ov_ov_composition']:.4f}, "
          f"OV-QK={vw['ov_qk_composition']:.4f}, score={vw['composition_score']:.4f}")

In [ ]:
# 4. Residual stream decomposition
decomp = residual_stream_decomposition(model, tokens)
print(f"Embed norm: {decomp['embed_norm']:.4f}")
for l in range(cfg.n_layers):
    print(f"Layer {l}: resid_norm={decomp['cumulative_norms'][l+1]:.4f}, "
          f"attn={decomp['attn_added_norms'][l]:.4f}, "
          f"mlp={decomp['mlp_added_norms'][l]:.4f}, "
          f"growth={decomp['growth_rate'][l]:.3f}")

In [ ]:
# 5. Path patching
corrupted = jnp.array([40, 41, 42, 43, 44, 45, 46, 47])
pp = path_patching_matrix(model, tokens, corrupted, metric)
print(f"Baseline: {pp['baseline_metric']:.4f}, Corrupted: {pp['corrupted_metric']:.4f}")
print(f"Most important: {pp['most_important_component']}")
print(f"\nAttention patching effects:")
for l in range(cfg.n_layers):
    vals = [f"{pp['attn_effects'][l, h]:.4f}" for h in range(cfg.n_heads)]
    print(f"  Layer {l}: {vals}")
print(f"MLP effects: {[f'{e:.4f}' for e in pp['mlp_effects']]}")